In [181]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [182]:
# creating consistant hashing
import hashlib
from bisect import bisect

In [183]:
# architecture of storage nodes
class StorageNode:
    # initialization
    def __init__(self, node: str, host: str):

        self.node = node
        self.host = host

In [184]:
# architecture of consistant hashing
class ConsistantHashing:
    # initialization
    def __init__(self, _keys: list[str], nodes: list[str], totalHashSpace: int):

        self._keys = _keys
        self.nodes = nodes

        self.totalHashSpace = totalHashSpace

    # hash function
    def hashFn(self, key: str, totalHashSpace: int) -> int:
    
        hash = hashlib.sha256()
    
        hash.update(bytes(key.encode("utf-8")))
        
        hex_hash = hash.hexdigest()
        
        return int(hex_hash, 16) % totalHashSpace # hash space range [0, totalHashSpace - 1]

    # add a storage node
    def addStorageNode(self, node):

        if len(self._keys) == self.totalHashSpace:

            raise Exception("Hash space is full")
            
        key = self.hashFn(node.host, self.totalHashSpace)
        
        index = bisect(self._keys, key) # right insertion

        # if node(key) is already present
        if index > 0 and self._keys[index - 1] == key:

            raise Exception("Node already present")

        self._keys.insert(index, key)
        self.nodes.insert(index, node)

        return True

    def removeStorageNode(self, node):

        if len(self._keys) == 0:

            raise Exception("No node is present")

            return

        key = self.hashFn(node.host, self.totalHashSpace)

        index = bisect(self._keys, key)

        if index > 0 and self._keys[index - 1] != key:

            raise Exception("Node is not present")
            return

        self._keys.pop(index - 1)
        self.nodes.pop(index - 1)
        
        return True


In [185]:
# let's say we allow only maximum 30 nodes
totalHashSpace = 10
# _keys = [-1] * totalHashSpace
# nodes = [-1] * totalHashSpace

_keys = []
nodes = []

cHashing = ConsistantHashing(_keys, nodes, totalHashSpace)

In [186]:
# Declare some storage nodes
node1 = StorageNode('M', "129.0.0.3")
node2 = StorageNode('B', "127.0.0.2")
node3 = StorageNode('Z', "198.0.0.3")
node4 = StorageNode('C', "198.0.0.9")

# add some storage nodes
cHashing.addStorageNode(node1)
cHashing.addStorageNode(node2)
cHashing.addStorageNode(node3)

# adding duplicate node
# cHashing.addStorageNode(node2)

# print(cHashing._keys)
# print(cHashing.nodes)

# adding the fourth node when the hash space is full
# cHashing.addStorageNode(node4)

print(cHashing._keys)

# delete some nodes
# cHashing.removeStorageNode(node1)
# print(cHashing._keys)

[1, 2, 7]
